In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy

from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


In [ ]:
def drop_unnecessary(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=["native-country", "education", "marital-status", "relationship", "race", "sex", "capital-loss", "capital-gain"])

In [ ]:
base_adult = pd.read_csv(filepath_or_buffer="adult\\adult.data")
base_adult

In [ ]:
base_adult["income"].value_counts()

In [ ]:
7841/32561

Age analysis

In [ ]:
base_adult["age"].describe()

In [ ]:
sns.histplot(data=base_adult["age"])

In [ ]:
# 16% of people up to 40yo have a networth above 50K (19.118 samples)
base_adult[base_adult.age <= 40]["income"].value_counts()

In [ ]:
# 37% of people on (40, 60]yo have a networth of 50k (11111 samples)
base_adult[(base_adult.age > 40) & (base_adult.age <= 60)]["income"].value_counts()

In [ ]:
# 23% of people above 69yo have a networth of 50k (2332 samples)
base_adult[base_adult.age > 60]["income"].value_counts()

applying normalization technique as suggested on https://www.datacamp.com/tutorial/normalization-vs-standardization

In [ ]:
age_min_max_scaler = MinMaxScaler()
base_adult["age"] = age_min_max_scaler.fit_transform(pd.DataFrame(base_adult["age"]))
base_adult

<h5>Native country analysis</h5>

In [ ]:
sns.countplot(base_adult["native-country"])

In [ ]:
np.unique(base_adult["income"], return_counts=True)

In [ ]:
sns.countplot(x=base_adult["income"])

In [ ]:
plt.hist(x=base_adult["age"])

In [ ]:
plt.hist(x=base_adult["education-num"])

In [ ]:
sns.countplot(y=base_adult["education"])

In [ ]:
sns.countplot(y=base_adult["marital-status"], hue=base_adult["income"])

In [ ]:
sns.countplot(y=base_adult["workclass"], hue=base_adult["income"])

In [ ]:
np.unique(base_adult["occupation"], return_counts=True)

In [ ]:
np.unique(base_adult[base_adult.workclass == " State-gov"]["occupation"], return_counts=True)

In [ ]:
np.unique(base_adult[base_adult.workclass == " Self-emp-not-inc"]["occupation"], return_counts=True)

In [ ]:
test_emp = base_adult["workclass"] + base_adult["occupation"]
test_emp

In [ ]:
np.unique(test_emp, return_counts=True)

In [ ]:
2740/test_emp.size

In [ ]:
test_workVSed = base_adult["workclass"] + base_adult["education"]
np.unique(test_workVSed, return_counts=True)

In [ ]:
pd.crosstab(index=base_adult["workclass"], columns=base_adult["income"])

In [ ]:
scipy.stats.contingency.association(pd.crosstab(index=base_adult["workclass"], columns=base_adult["income"]))

In [ ]:
scipy.stats.contingency.association(pd.crosstab(index=base_adult["workclass"], columns=base_adult["income"]),
                                    method="tschuprow")

In [ ]:
scipy.stats.contingency.association(pd.crosstab(index=base_adult["workclass"], columns=base_adult["income"]),
                                    method="pearson")

Dropping values `?` from dataset

In [ ]:
base_adult[base_adult.workclass == " ?"]

In [ ]:
base_adult.drop(base_adult.loc[base_adult.workclass == " ?"].index, inplace=True)

In [ ]:
base_adult.drop(base_adult.loc[base_adult.occupation == " ?"].index, inplace=True)

In [ ]:
base_adult

In [ ]:
np.unique(base_adult["occupation"], return_counts=True)

In [ ]:
# [Adm-clerical, Exec-managerial, Prof-specialty, Sales, Tech-support] -> Office-Worker
# [Armed-Forces, Protective-serv] -> Security-Worker
# [Craft-repair, Machine-op-inspct, Transport-moving] -> Machinery-Worker
# [Farming-fishing, Handlers-cleaners, Priv-house-serv] -> Manual-Worker
# Other-Service
def occupation_mapping(occupation: str) -> str:
    if occupation.strip() in ["Adm-clerical", "Exec-managerial", "Prof-specialty", "Sales", "Tech-support"]:
        return "Office-Worker"
    elif occupation.strip() in ["Armed-Forces", "Protective-serv"]:
        return "Security-Worker"
    elif occupation.strip() in ["Craft-repair", "Machine-op-inspct", "Transport-moving"]:
        return "Machinery-Worker"
    elif occupation.strip() in ["Farming-fishing", "Handlers-cleaners", "Priv-house-serv"]:
        return "Manual-Worker"
    elif occupation.strip() == "Other-service":
        return "Other-Service"


In [ ]:
base_adult["occupation"] = base_adult["occupation"].apply(func=occupation_mapping)
base_adult

Dropping other categorical variables

In [ ]:
np.unique(base_adult["workclass"], return_counts=True)

In [ ]:
def summarize_workclass(workclass: str) -> str:
    if workclass.strip() in ["Federal-gov", "Local-gov", "State-gov"]:
        return "GOV"
    elif workclass.strip() in ["Private", "Self-emp-inc", "Self-emp-not-inc"]:
        return "PRIVATE"
    else:
        return "UNPAID"

In [ ]:
base_adult["workclass"] = base_adult["workclass"].apply(func=summarize_workclass)
base_adult

In [ ]:
np.unique(base_adult["capital-loss"], return_counts=True)

In [ ]:
np.unique_counts(base_adult[base_adult["capital-loss"] == 0]["income"])

In [ ]:
np.unique_counts(base_adult[base_adult["capital-loss"] != 0]["income"])

In [ ]:
np.unique(base_adult["capital-gain"], return_counts=True)

In [ ]:
np.unique_counts(base_adult[base_adult["capital-gain"] == 0]["income"])

In [ ]:
np.unique_counts(base_adult[base_adult["capital-gain"] != 0]["income"])

In [ ]:
base_adult = drop_unnecessary(base_adult)
base_adult

In [ ]:
base_adult = pd.get_dummies(base_adult, columns=["workclass", "occupation"], dtype=float)
base_adult

In [ ]:
fnlwgt_scaler = MinMaxScaler()
base_adult["fnlwgt"] = fnlwgt_scaler.fit_transform(pd.DataFrame(base_adult["fnlwgt"]))

edu_scaler = MinMaxScaler()
base_adult["education-num"] = edu_scaler.fit_transform(pd.DataFrame(base_adult["education-num"]))

hours_scaler = MinMaxScaler()
base_adult["hours-per-week"] = hours_scaler.fit_transform(pd.DataFrame(base_adult["hours-per-week"]))

base_adult

In [ ]:
X = base_adult.drop(columns=["income"])
y = base_adult["income"]

In [ ]:
income_replacer = {" <=50K": 0, " >50K": 1}
y.replace(income_replacer, inplace=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [ ]:
neigh_class = KNeighborsClassifier()
neigh_class.fit(X_train, y_train)

In [ ]:
#tree_clf = DecisionTreeClassifier(random_state=0)

In [ ]:
#cross_val_score(tree_clf, X, y, cv=10)
#tree_clf.fit(X=X, y=y)